# Exercise 4 — Train ResNet on Kaggle GPU

This notebook clones the repo, sets up the data, runs the tests, trains the solar-cell defect classifier, and exports an ONNX model for the leaderboard.

## Step 0 — Verify the GPU
Confirms PyTorch can see a CUDA GPU. If `cuda available: False`, stop and enable the GPU accelerator in Settings — training on CPU here is impractically slow.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
!nvidia-smi -L

## Step 1 — Clone the repository
Pulls a fresh copy of the repo into Kaggle's writable `/kaggle/working` directory and moves into the exercise folder. `--depth 1` fetches only the latest commit (faster, smaller), and the `rm -rf` first makes re-runs start clean. Requires **Internet → On** and your latest code pushed to GitHub.

In [ ]:
%cd /kaggle/working
!rm -rf deep-learning
!git clone --depth 1 https://github.com/mo-karbalaee/deep-learning.git
%cd /kaggle/working/deep-learning/ex4/src_to_implement

## Step 2 — Install dependencies
Kaggle images already ship torch, torchvision, scikit-image, pandas and scikit-learn. We only add `onnxruntime` (runs the ONNX part of the test suite) and `onnxscript` (required by torch 2.x's ONNX exporter). The import line fails fast if anything essential is missing.

In [ ]:
!pip -q install onnxruntime onnxscript
import skimage, pandas, sklearn, torchvision
print('deps OK')

## Step 3 — Extract the dataset
`images.zip` is committed in the repo; this unzips it into `images/` (~2000 electroluminescence PNGs) next to `data.csv`, which is where `ChallengeDataset` expects to find them. It is skipped automatically if the folder already exists.

In [ ]:
import zipfile, os
if not os.path.isdir('images'):
    zipfile.ZipFile('images.zip').extractall('.')
print(len(os.listdir('images')), 'images extracted')

## Step 4 — Run the unit tests (optional)
Runs the graded suite with the `Bonus` flag: `TestDataset` (data pipeline shape + normalization) and `TestModel` (forward pass + ONNX export/reload). This verifies *code correctness* only — an untrained model passes. Safe to skip if you just want to train.

In [ ]:
!python PytorchChallengeTests.py Bonus

## Step 4.5 — Opset probe (DO THIS FIRST)\nWe bumped the ONNX opset to 17 so we can use modern backbones (convnext, efficientnet). But we do **not** know the leaderboard accepts opset 17. This cell exports an **untrained** ensemble to `probe.onnx` (a few minutes, just downloads weights + exports). Submit `probe.onnx` to the leaderboard: it will score ~random, but if the server **accepts** it, opset 17 works and you can safely run the long training below. If the server **rejects** it, stop — revert to the opset-10 resnet ensemble instead of wasting 2-3 hours training.

In [ ]:
import torch as t, model, shutil, os
from trainer import Trainer
probe_members = [model.build_convnext(), model.build_efficientnet(), model.build_resnext50()]
probe = model.Ensemble(probe_members)
Trainer(probe, t.nn.BCELoss(), cuda=False).save_onnx('probe.onnx')
shutil.copy('probe.onnx', '/kaggle/working/probe.onnx')
print('probe.onnx exported OK ({:.0f} MB) - submit it to check opset-17 acceptance'.format(os.path.getsize('probe.onnx') / 1e6))

## Step 5 — Train the diverse ensemble (only after the probe is accepted)\nTrains three modern backbones (convnext_tiny, efficientnet_b3, resnext50), one per cross-validation fold, each fine-tuned with class-imbalance oversampling, a dropout head, `BCELoss`, Adam, LR scheduler and early stopping. They are wrapped into one `Ensemble` that averages their sigmoids with test-time augmentation (original + h-flip + v-flip) and exported to `ensemble.onnx` at opset 17.\n\nSlow step: three modern backbones back to back, roughly 2-3 hours on a T4. Do not run this until `probe.onnx` was accepted by the leaderboard.

In [ ]:
!python train.py

## Step 6 — Collect the ensemble ONNX\n`train.py` already exported the averaged 3-model, TTA ensemble to `ensemble.onnx`. This copies it into `/kaggle/working/` for download and upload to the leaderboard.\n\nNote: with three backbones this file is roughly 350-400 MB. If the leaderboard rejects it for size, drop one member in `train.py` (e.g. remove resnet101) or fall back to a single backbone.

In [ ]:
import os, shutil
assert os.path.exists('ensemble.onnx'), 'ensemble.onnx not found - did train.py finish?'
shutil.copy('ensemble.onnx', '/kaggle/working/ensemble.onnx')
size_mb = os.path.getsize('ensemble.onnx') / 1e6
print('Saved /kaggle/working/ensemble.onnx ({:.0f} MB)'.format(size_mb))